In [17]:
import pandas as pd
import time
import io
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- CONFIGURACIÓN ---
URL_CAPOLOGY = "https://capology.com/fr/ligue-1/salaries/"
NOMBRE_SALARIOS_RAW = "capology_ligue1_25_26_raw.csv"
CARPETA = r"c:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Salarios"

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def extraccion_con_navegacion_forzada(url):
    try:
        driver.get(url)
        print("\n" + "="*50)
        print("CONFIGURACIÓN MANUAL")
        print("1. Pon USD y NET.")
        print("2. Cierra cualquier anuncio (X).")
        print("="*50)
        input("Presiona ENTER cuando la tabla esté lista...")

        lista_dataframes = []
        nombres_vistos = set()
        page_count = 1

        while True:
            # Espera a que la tabla cargue
            wait = WebDriverWait(driver, 15)
            wait.until(EC.presence_of_element_located((By.ID, "table")))
            
            # Capturar primer nombre para el control de bucle
            try:
                # Damos un segundo para que el texto cargue bien
                time.sleep(1)
                primer_nombre = driver.find_element(By.CSS_SELECTOR, "#table tbody tr td a").text
            except:
                primer_nombre = "VACIO"

            # SI EL NOMBRE YA EXISTE, REALMENTE NO CAMBIÓ DE PÁGINA
            if primer_nombre in nombres_vistos:
                print(f"Contenido repetido ({primer_nombre}). Finalizando.")
                break
            
            print(f"Extrayendo página {page_count}...")
            
            # Captura de datos
            html_tabla = driver.find_element(By.ID, "table").get_attribute('outerHTML')
            df_temp = pd.read_html(io.StringIO(html_tabla))[0]
            
            if not df_temp.empty:
                lista_dataframes.append(df_temp)
                nombres_vistos.add(primer_nombre)

            # --- INTENTO DE NAVEGACIÓN REFORZADO ---
            try:
                # Intentamos buscar el botón 'Next' por diferentes XPaths comunes en Capology
                next_selectors = [
                    "//li[contains(@class, 'next')]//a",
                    "//a[contains(text(), 'Next')]",
                    "//a[normalize-space()='Next']"
                ]
                
                encontrado = False
                for selector in next_selectors:
                    btn = driver.find_elements(By.XPATH, selector)
                    if btn:
                        # Verificamos que el botón no esté deshabilitado (clase 'disabled')
                        parent_class = btn[0].find_element(By.XPATH, "..").get_attribute("class")
                        if "disabled" not in parent_class:
                            # Hacemos scroll y clic por JS para ignorar anuncios
                            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn[0])
                            time.sleep(1)
                            driver.execute_script("arguments[0].click();", btn[0])
                            encontrado = True
                            break
                
                if encontrado:
                    page_count += 1
                    time.sleep(4) # Espera a que la tabla cambie
                else:
                    print("No se encontró un botón 'Next' activo. Fin de la tabla.")
                    break
            except Exception as e:
                print(f"Error al intentar navegar: {e}")
                break

        return pd.concat(lista_dataframes, ignore_index=True) if lista_dataframes else None

    except Exception as e:
        print(f"Error en el proceso: {e}")
        return None

try:
    df_resultado = extraccion_con_navegacion_forzada(URL_CAPOLOGY)
    if df_resultado is not None:
        if not os.path.exists(CARPETA): os.makedirs(CARPETA)
        ruta = os.path.join(CARPETA, NOMBRE_SALARIOS_RAW)
        
        # Limpiar filas de basura
        df_resultado = df_resultado[df_resultado.iloc[:, 0].astype(str).str.upper() != 'PLAYER']
        
        df_resultado.drop_duplicates().to_csv(ruta, index=False, encoding='utf-8-sig')
        print(f"\n¡LOGRADO! Se guardaron {len(df_resultado)} registros en {ruta}")
finally:
    driver.quit()


CONFIGURACIÓN MANUAL
1. Pon USD y NET.
2. Cierra cualquier anuncio (X).
Extrayendo página 1...
Extrayendo página 2...
Extrayendo página 3...
Extrayendo página 4...
Extrayendo página 5...
Extrayendo página 6...
Extrayendo página 7...
Extrayendo página 8...
Extrayendo página 9...
Extrayendo página 10...
Extrayendo página 11...
Extrayendo página 12...
Extrayendo página 13...
Extrayendo página 14...
Extrayendo página 15...
Extrayendo página 16...
Extrayendo página 17...
Extrayendo página 18...
Extrayendo página 19...
Extrayendo página 20...
Extrayendo página 21...
Contenido repetido (Ousmane Dembélé). Finalizando.

¡LOGRADO! Se guardaron 502 registros en c:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Salarios\capology_ligue1_25_26_raw.csv
